# CS781 Phase 1 — Generation only (decoding experiments, no training)

Loads an **already trained** model (Arch 1 Llama-style decoder, own BPE tokenizer) and generates the
1,000 test continuations with one or more **decoding settings**. Each setting produces its own
`submission.zip`, plus diagnostics (repetition, short outputs) so you can compare settings before
submitting to Codabench.

**How to use (Kaggle):**
1. Make a dataset from the three files in the trained model folder `legal-decoder_llama-final/`:
   `pytorch_model.bin`, `config.json`, `legal_bpe_tokenizer.json`. Add it as input.
2. Accelerator **GPU T4 x2** (one GPU is used), Internet **on**, secret `HF` ticked.
3. Set `MODEL_DIR` (run `!find /kaggle/input -name config.json`) and edit `DECODING_CONFIGS`.
4. *Save Version → Save & Run All*. Outputs: `/kaggle/working/generation/<setting>/submission.zip`
   and `/kaggle/working/generation/summary.json`.


## 1. Hugging Face login (the test prompts are downloaded from the course dataset)

In [ ]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

## 2. Configuration

In [ ]:
import os, json, time, math, zipfile, random, statistics
from collections import defaultdict
import numpy as np
import torch

MODEL_DIR = None

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TEST_FILE = "Phase_1_Pretraining/Test/test.json"

SEED = 42
MODEL_CONTEXT_LENGTH = 1024
TARGET_GENERATION_WORDS = 128
MAX_GENERATION_TOKENS = 320
GENERATION_BATCH_SIZE = 32
MAX_TEST_EXAMPLES = None

DECODING_CONFIGS = [
    dict(name="T07_p09_rep12",        temperature=0.7, top_p=0.9, top_k=0, repetition_penalty=1.2, no_repeat_ngram=0),
    dict(name="T07_p09_rep11_nr6",    temperature=0.7, top_p=0.9, top_k=0, repetition_penalty=1.1, no_repeat_ngram=6),
    dict(name="T09_p09_rep12_nr6",    temperature=0.9, top_p=0.9, top_k=0, repetition_penalty=1.2, no_repeat_ngram=6),
    dict(name="greedy_rep12_nr6",     temperature=0.0, top_p=1.0, top_k=0, repetition_penalty=1.2, no_repeat_ngram=6),
]

OUTPUT_ROOT = "/kaggle/working/generation"
os.makedirs(OUTPUT_ROOT, exist_ok=True)
assert MODEL_DIR and os.path.exists(os.path.join(MODEL_DIR, "pytorch_model.bin")), \
    "Set MODEL_DIR to the folder containing pytorch_model.bin (run: !find /kaggle/input -name pytorch_model.bin)"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "| configs:", [c["name"] for c in DECODING_CONFIGS])

## 3. Our BPE tokenizer (same code as in training)

In [ ]:
import heapq
import json
import os
import re
import time
from collections import Counter, defaultdict

PRETOKENIZE_PATTERN = (
    r"'(?:[sdmt]|ll|ve|re)| ?[^\W\d_]+| ?\d{1,3}| ?(?:[^\s\w]|_)+|\s+(?!\S)|\s+"
)
EOS_TOKEN = "<|endoftext|>"

class LegalBPETokenizer:
    def __init__(self, merges=None, name="legal-bpe"):
        self.name = name
        self.merges = [tuple(m) for m in (merges or [])]
        self._pattern = re.compile(PRETOKENIZE_PATTERN)
        self._build_tables()

    def _build_tables(self):
        self.ranks = {pair: i for i, pair in enumerate(self.merges)}
        self.vocab = {i: bytes([i]) for i in range(256)}
        for i, (a, b) in enumerate(self.merges):
            self.vocab[256 + i] = self.vocab[a] + self.vocab[b]
        self.eos_token = EOS_TOKEN
        self.eos_token_id = 256 + len(self.merges)
        self.pad_token = self.eos_token
        self.pad_token_id = self.eos_token_id
        self.model_max_length = 10**9
        self._cache = {}

    def __len__(self):
        return 256 + len(self.merges) + 1

    @property
    def vocab_size(self):
        return len(self)

    def pretokenize(self, text):
        return self._pattern.findall(text)

    @classmethod
    def train(cls, texts, vocab_size, name="legal-bpe", min_frequency=2, verbose=True):
        assert 258 <= vocab_size <= 65535, "vocab_size must fit in uint16"
        t0 = time.time()
        pattern = re.compile(PRETOKENIZE_PATTERN)

        word_counts = Counter()
        n_chars = 0
        for text in texts:
            n_chars += len(text)
            word_counts.update(pattern.findall(text))
        if verbose:
            print(f"[BPE] {n_chars:,} characters, {len(word_counts):,} unique words "
                  f"({time.time() - t0:.0f}s)")

        words = [list(w.encode("utf-8")) for w in word_counts]
        freqs = list(word_counts.values())
        del word_counts

        pair_counts = defaultdict(int)
        pair_where = defaultdict(set)
        for wi, (w, f) in enumerate(zip(words, freqs)):
            for a, b in zip(w, w[1:]):
                pair_counts[(a, b)] += f
                pair_where[(a, b)].add(wi)
        heap = [(-c, p) for p, c in pair_counts.items()]
        heapq.heapify(heap)

        n_merges = vocab_size - 257
        merges = []
        while len(merges) < n_merges and heap:
            neg_c, pair = heapq.heappop(heap)
            count = pair_counts.get(pair, 0)
            if count <= 0:
                continue
            if -neg_c != count:
                heapq.heappush(heap, (-count, pair))
                continue
            if count < min_frequency:
                if verbose:
                    print(f"[BPE] stopping early: best pair frequency {count} < {min_frequency}")
                break

            new_id = 256 + len(merges)
            merges.append(pair)
            a, b = pair
            changed = set()
            for wi in list(pair_where[pair]):
                w, f = words[wi], freqs[wi]
                for p in zip(w, w[1:]):
                    pair_counts[p] -= f
                    changed.add(p)
                new_w, i = [], 0
                while i < len(w):
                    if i + 1 < len(w) and w[i] == a and w[i + 1] == b:
                        new_w.append(new_id)
                        i += 2
                    else:
                        new_w.append(w[i])
                        i += 1
                words[wi] = new_w
                for p in zip(new_w, new_w[1:]):
                    pair_counts[p] += f
                    pair_where[p].add(wi)
                    changed.add(p)
            del pair_where[pair]
            pair_counts.pop(pair, None)
            changed.discard(pair)
            for p in changed:
                c = pair_counts.get(p, 0)
                if c > 0:
                    heapq.heappush(heap, (-c, p))
                else:
                    pair_counts.pop(p, None)
                    pair_where.pop(p, None)
            if verbose and (len(merges) % 2000 == 0):
                print(f"[BPE] {len(merges):,}/{n_merges:,} merges ({time.time() - t0:.0f}s)")

        tok = cls(merges, name=name)
        if verbose:
            print(f"[BPE] done: vocab size {len(tok):,} ({time.time() - t0:.0f}s)")
        return tok

    def _encode_word(self, word):
        ids = self._cache.get(word)
        if ids is not None:
            return ids
        ids = list(word.encode("utf-8"))
        ranks = self.ranks
        while len(ids) >= 2:
            best_rank, best_pair = None, None
            for p in zip(ids, ids[1:]):
                r = ranks.get(p)
                if r is not None and (best_rank is None or r < best_rank):
                    best_rank, best_pair = r, p
            if best_pair is None:
                break
            a, b = best_pair
            new_id = 256 + best_rank
            merged, i = [], 0
            while i < len(ids):
                if i + 1 < len(ids) and ids[i] == a and ids[i + 1] == b:
                    merged.append(new_id)
                    i += 2
                else:
                    merged.append(ids[i])
                    i += 1
            ids = merged
        if len(self._cache) > 2_000_000:
            self._cache.clear()
        self._cache[word] = ids
        return ids

    def encode(self, text, add_special_tokens=False):
        out = []
        for word in self._pattern.findall(text):
            out.extend(self._encode_word(word))
        return out

    def __call__(self, texts, add_special_tokens=False, **kwargs):
        if isinstance(texts, str):
            return {"input_ids": self.encode(texts)}
        return {"input_ids": [self.encode(t) for t in texts]}

    def decode(self, ids, skip_special_tokens=True, **kwargs):
        parts = []
        for i in ids:
            i = int(i)
            if i == self.eos_token_id:
                if not skip_special_tokens:
                    parts.append(self.eos_token.encode("utf-8"))
                continue
            b = self.vocab.get(i)
            if b is not None:
                parts.append(b)
        return b"".join(parts).decode("utf-8", errors="replace")

    def batch_decode(self, sequences, skip_special_tokens=True, **kwargs):
        return [self.decode(s, skip_special_tokens=skip_special_tokens) for s in sequences]

    def convert_ids_to_tokens(self, ids):
        return [self.vocab[i].decode("utf-8", errors="replace") if i in self.vocab else self.eos_token
                for i in ids]

    def save_pretrained(self, directory):
        os.makedirs(directory, exist_ok=True)
        path = os.path.join(directory, "legal_bpe_tokenizer.json")
        with open(path, "w") as f:
            json.dump({
                "type": "byte-level-bpe",
                "name": self.name,
                "pattern": PRETOKENIZE_PATTERN,
                "eos_token": self.eos_token,
                "eos_token_id": self.eos_token_id,
                "vocab_size": len(self),
                "merges": [list(m) for m in self.merges],
            }, f)
        return path

    @classmethod
    def from_pretrained(cls, directory):
        path = directory if directory.endswith(".json") else os.path.join(directory, "legal_bpe_tokenizer.json")
        with open(path) as f:
            data = json.load(f)
        assert data["pattern"] == PRETOKENIZE_PATTERN, "tokenizer was saved with a different pre-tokenisation pattern"
        tok = cls(data["merges"], name=data.get("name", "legal-bpe"))
        assert tok.eos_token_id == data["eos_token_id"]
        return tok

tokenizer = LegalBPETokenizer.from_pretrained(MODEL_DIR)
EOS_ID = tokenizer.eos_token_id
REAL_VOCAB = len(tokenizer)
print("Tokenizer:", tokenizer.name, "| vocab:", REAL_VOCAB)

## 4. Model code (identical to the training notebook) and trained weights

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        xf = x.float()
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (xf * self.weight.float()).type_as(x)

class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, max_positions, theta=10_000.0):
        super().__init__()
        inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim))
        freqs = torch.outer(torch.arange(max_positions, dtype=torch.float32), inv_freq)
        self.register_buffer("cos_cached", torch.cat([freqs.cos(), freqs.cos()], dim=-1), persistent=False)
        self.register_buffer("sin_cached", torch.cat([freqs.sin(), freqs.sin()], dim=-1), persistent=False)

    def forward(self, x, offset=0):
        T = x.size(-2)
        cos = self.cos_cached[offset:offset + T].to(x.dtype)
        sin = self.sin_cached[offset:offset + T].to(x.dtype)
        x1, x2 = x.chunk(2, dim=-1)
        return x * cos + torch.cat((-x2, x1), dim=-1) * sin

class SelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, dropout=0.0):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head, self.head_dim = n_head, n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        self.dropout = dropout

    def forward(self, x, rope, attn_mask=None, is_causal=True, cache=None, offset=0):
        B, T, C = x.shape
        q, k, v = self.qkv(x).view(B, T, 3, self.n_head, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k = rope(q, offset), rope(k, offset)
        if cache is not None:
            if "k" in cache:
                k = torch.cat([cache["k"], k], dim=2)
                v = torch.cat([cache["v"], v], dim=2)
            cache["k"], cache["v"] = k, v
        y = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=attn_mask,
            is_causal=is_causal and attn_mask is None,
            dropout_p=self.dropout if self.training else 0.0,
        )
        return self.proj(y.transpose(1, 2).reshape(B, T, C))

class SwiGLU(nn.Module):
    def __init__(self, n_embd, hidden):
        super().__init__()
        self.w_gate_up = nn.Linear(n_embd, 2 * hidden, bias=False)
        self.w_down = nn.Linear(hidden, n_embd, bias=False)

    def forward(self, x):
        gate, up = self.w_gate_up(x).chunk(2, dim=-1)
        return self.w_down(F.silu(gate) * up)

class DecoderBlock(nn.Module):
    def __init__(self, n_embd, n_head, ffn_hidden, dropout=0.0):
        super().__init__()
        self.norm1 = RMSNorm(n_embd)
        self.attn = SelfAttention(n_embd, n_head, dropout)
        self.norm2 = RMSNorm(n_embd)
        self.mlp = SwiGLU(n_embd, ffn_hidden)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, rope, attn_mask=None, is_causal=True, cache=None, offset=0):
        x = x + self.drop(self.attn(self.norm1(x), rope, attn_mask, is_causal, cache, offset))
        x = x + self.drop(self.mlp(self.norm2(x)))
        return x

def init_weights(model, n_layers, std=0.02):
    for module in model.modules():
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=std)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=std)
    for name, p in model.named_parameters():
        if name.endswith("proj.weight") or name.endswith("w_down.weight"):
            nn.init.normal_(p, mean=0.0, std=std / math.sqrt(2 * n_layers))

def lm_loss(lm_head, hidden, input_ids, loss_start=0):
    start = max(int(loss_start), 1)
    logits = lm_head(hidden[:, start - 1:-1])
    targets = input_ids[:, start:]
    return F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))

In [ ]:
class DecoderOnlyLM(nn.Module):

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        d, L, H = cfg["n_embd"], cfg["n_layer"], cfg["n_head"]
        self.embed = nn.Embedding(cfg["vocab_size"], d)
        self.drop = nn.Dropout(cfg["dropout"])
        self.rope = RotaryEmbedding(d // H, cfg["max_positions"], cfg["rope_theta"])
        self.blocks = nn.ModuleList(
            [DecoderBlock(d, H, cfg["ffn_hidden"], cfg["dropout"]) for _ in range(L)]
        )
        self.norm = RMSNorm(d)
        self.lm_head = nn.Linear(d, cfg["vocab_size"], bias=False)
        self.lm_head.weight = self.embed.weight
        init_weights(self, L)

    def _hidden(self, input_ids, caches=None, offset=0, is_causal=True):
        x = self.drop(self.embed(input_ids))
        for i, block in enumerate(self.blocks):
            x = block(x, self.rope, is_causal=is_causal,
                      cache=None if caches is None else caches[i], offset=offset)
        return self.norm(x)

    def forward(self, input_ids, prefix_len=0):
        return lm_loss(self.lm_head, self._hidden(input_ids), input_ids, prefix_len)

    @torch.no_grad()
    def prefill(self, prompt_ids):
        caches = [dict() for _ in self.blocks]
        h = self._hidden(prompt_ids, caches, offset=0, is_causal=True)
        return self.lm_head(h[:, -1]), {"caches": caches, "pos": prompt_ids.size(1)}

    @torch.no_grad()
    def decode_step(self, token_ids, state):
        h = self._hidden(token_ids[:, None], state["caches"], offset=state["pos"], is_causal=False)
        state["pos"] += 1
        return self.lm_head(h[:, -1]), state

    @torch.no_grad()
    def teacher_forced_logits(self, prompt_ids, cont_ids):
        ids = torch.cat([prompt_ids, cont_ids], dim=1)
        h = self._hidden(ids)
        return self.lm_head(h[:, prompt_ids.size(1) - 1:-1])

MODEL_CLASS = DecoderOnlyLM

with open(os.path.join(MODEL_DIR, "config.json")) as f:
    model_config = json.load(f)
assert model_config.get("arch") == "decoder_llama", f"This notebook is for the decoder_llama model, got {model_config.get('arch')}"
model = MODEL_CLASS(model_config)
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, "pytorch_model.bin"), map_location="cpu"))
model = model.to(device).eval()
print(f"Loaded {model_config['arch']}: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M parameters")

PROMPT_TOKENS = min(MODEL_CONTEXT_LENGTH, model_config["max_positions"]) - MAX_GENERATION_TOKENS
assert PROMPT_TOKENS > 0
print("Prompt tokens kept:", PROMPT_TOKENS, "| generation budget:", MAX_GENERATION_TOKENS)

## 5. Test prompts

In [ ]:
from huggingface_hub import hf_hub_download

test_file_path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=TEST_FILE, repo_type="dataset", token=HF_TOKEN)
with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)
items = list(test_data.items())
if MAX_TEST_EXAMPLES:
    items = items[:MAX_TEST_EXAMPLES]
print("Test examples:", len(items))

def prepare_prompt_ids(text):
    ids = tokenizer.encode(text) or [EOS_ID]
    return ids[-PROMPT_TOKENS:]

buckets = defaultdict(list)
for example_id, record in items:
    ids = prepare_prompt_ids(record["text"])
    buckets[len(ids)].append((str(example_id), ids))
batches = []
for length in sorted(buckets, reverse=True):
    group = buckets[length]
    for start in range(0, len(group), GENERATION_BATCH_SIZE):
        batches.append(group[start:start + GENERATION_BATCH_SIZE])
print(f"{len(batches)} batches ({len(buckets)} distinct prompt lengths)")

## 6. Decoding
#### Temperature / top-p / top-k / repetition penalty as in the training notebook, plus optional no-repeat n-gram blocking.

In [ ]:
import contextlib

class NgramBlocker:
    def __init__(self, batch_size, n):
        self.n = n
        self.seqs = [[] for _ in range(batch_size)]
        self.seen = [defaultdict(set) for _ in range(batch_size)]

    def banned(self):
        if self.n <= 0:
            return None
        out = []
        for seq, seen in zip(self.seqs, self.seen):
            if len(seq) >= self.n - 1:
                out.append(seen.get(tuple(seq[len(seq) - self.n + 1:]), ()))
            else:
                out.append(())
        return out

    def update(self, tokens):
        for b, t in enumerate(tokens):
            seq = self.seqs[b]
            seq.append(t)
            if self.n > 0 and len(seq) >= self.n:
                self.seen[b][tuple(seq[-self.n:-1])].add(seq[-1])

def sample_next_token(logits, generated, cfg, banned=None):
    logits = logits.float()
    if logits.size(-1) > REAL_VOCAB:
        logits[:, REAL_VOCAB:] = float("-inf")
    logits[:, EOS_ID] = float("-inf")
    if banned is not None:
        for b, toks in enumerate(banned):
            if toks:
                logits[b, list(toks)] = float("-inf")
    rp = cfg["repetition_penalty"]
    if generated is not None and rp != 1.0:
        prev = logits.gather(1, generated)
        prev = torch.where(prev > 0, prev / rp, prev * rp)
        logits = logits.scatter(1, generated, prev)
    if cfg["temperature"] <= 0:
        return logits.argmax(dim=-1)
    logits = logits / cfg["temperature"]
    if cfg["top_k"] > 0:
        kth = torch.topk(logits, cfg["top_k"], dim=-1).values[:, -1:]
        logits = logits.masked_fill(logits < kth, float("-inf"))
    if cfg["top_p"] < 1.0:
        sorted_logits, sorted_idx = torch.sort(logits, descending=True, dim=-1)
        probs = sorted_logits.softmax(dim=-1)
        remove = probs.cumsum(dim=-1) - probs > cfg["top_p"]
        sorted_logits = sorted_logits.masked_fill(remove, float("-inf"))
        logits = torch.full_like(logits, float("-inf")).scatter(1, sorted_idx, sorted_logits)
    return torch.multinomial(logits.softmax(dim=-1), num_samples=1).squeeze(1)

def enough_words(generated):
    texts = tokenizer.batch_decode(generated.tolist())
    return all(len(t.split()) > TARGET_GENERATION_WORDS for t in texts)

@torch.inference_mode()
def generate_batch(prompt_id_lists, cfg):
    input_ids = torch.tensor(prompt_id_lists, dtype=torch.long, device=device)
    amp = torch.autocast(device_type="cuda", dtype=torch.float16) if device.type == "cuda" else contextlib.nullcontext()
    blocker = NgramBlocker(len(prompt_id_lists), cfg["no_repeat_ngram"])
    with amp:
        logits, state = model.prefill(input_ids)
        generated = []
        for step in range(MAX_GENERATION_TOKENS):
            prev = torch.stack(generated, dim=1) if generated else None
            next_token = sample_next_token(logits, prev, cfg, blocker.banned())
            generated.append(next_token)
            blocker.update(next_token.tolist())
            if (step + 1) % 16 == 0 and enough_words(torch.stack(generated, dim=1)):
                break
            if step + 1 < MAX_GENERATION_TOKENS:
                logits, state = model.decode_step(next_token, state)
    out = []
    for row in torch.stack(generated, dim=1).tolist():
        words = tokenizer.decode(row).split()[:TARGET_GENERATION_WORDS]
        out.append(" ".join(words))
    return out

def diagnostics(texts):
    def rep4(t):
        w = t.split(); g = [tuple(w[i:i + 4]) for i in range(len(w) - 3)]
        return 1 - len(set(g)) / max(1, len(g))
    r = sorted(rep4(t) for t in texts)
    allw = " ".join(texts).split()
    bigrams = list(zip(allw, allw[1:]))
    wc = [len(t.split()) for t in texts]
    return {
        "rep4_median": round(statistics.median(r), 3),
        "rep4_p90": round(r[int(0.9 * (len(r) - 1))], 3),
        "outputs_over_30pct_repeated": sum(x > 0.3 for x in r),
        "outputs_under_128_words": sum(x < TARGET_GENERATION_WORDS for x in wc),
        "min_words": min(wc),
        "distinct_bigram_ratio": round(len(set(bigrams)) / max(1, len(bigrams)), 3),
    }

## 7. Generate one submission per decoding setting

In [ ]:
summary = []
for cfg in DECODING_CONFIGS:
    torch.manual_seed(SEED); random.seed(SEED); np.random.seed(SEED)
    out_dir = os.path.join(OUTPUT_ROOT, cfg["name"])
    os.makedirs(out_dir, exist_ok=True)
    print(f"\n=== {cfg['name']}: {cfg}")
    predictions, t0, done = {}, time.time(), 0
    for bi, batch in enumerate(batches):
        texts = generate_batch([ids for _, ids in batch], cfg)
        for (example_id, _), text in zip(batch, texts):
            predictions[example_id] = {"text": text}
        done += len(batch)
        if bi % 10 == 0 or done == len(items):
            print(f"  {done}/{len(items)} | {time.time() - t0:.0f}s")
    missing = set(str(k) for k, _ in items) - set(predictions)
    assert not missing, f"{len(missing)} test ids have no prediction"

    pred_path = os.path.join(out_dir, "prediction.json")
    with open(pred_path, "w", encoding="utf-8") as f:
        json.dump(predictions, f, ensure_ascii=False, indent=2)
    zip_path = os.path.join(out_dir, "submission.zip")
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(pred_path, arcname="prediction.json")

    d = diagnostics([p["text"] for p in predictions.values()])
    d.update(name=cfg["name"], seconds=round(time.time() - t0), config=cfg,
             valid_submission=(len(predictions) == len(test_data)))
    summary.append(d)
    print("  diagnostics:", {k: v for k, v in d.items() if k not in ("config",)})
    print("  saved:", zip_path)
    first = items[0][0]
    print("  sample:", predictions[str(first)]["text"][:400])

with open(os.path.join(OUTPUT_ROOT, "summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

## 8. Summary
#### For reference, the baseline (temperature 0.1, top-p 0.95, repetition penalty 1.1) had: median repeated 4-grams 0.376, 592 outputs >30% repeated, 15 outputs <128 words, distinct-bigram ratio 0.184, BERTScore 0.8234.

In [ ]:
cols = ["name", "rep4_median", "rep4_p90", "outputs_over_30pct_repeated", "outputs_under_128_words", "min_words", "distinct_bigram_ratio", "seconds"]
print(" | ".join(cols))
for d in summary:
    print(" | ".join(str(d[c]) for c in cols))
print("\nSubmit the submission.zip of the settings you want from", OUTPUT_ROOT)